# Inspect WVS Option Tokens

This notebook mirrors the prompt format in `scripts/analysis/evaluate_wvs.py` and checks whether a model can actually emit `A/B/C/D` as the first completion token.

For each example question it:
- builds the exact multiple-choice prompt used in WVS eval
- prints the tokenizer's single-token encodings for `A/B/C/D`
- prints the top next tokens after the prompt
- prints the raw and normalized `A/B/C/D` probabilities used by the evaluator
- shows a short greedy continuation


In [1]:
from __future__ import annotations

import inspect
import json
import os
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "dempo").exists() else cwd.parent
assert (REPO_ROOT / "dempo").exists(), f"Could not find repo root from {cwd}"
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

OPTION_LABELS = ["A", "B", "C", "D"]


In [2]:
# Edit these paths as needed.
MODEL_IDS = [
    "meta-llama/Llama-3.1-8B",
    "/workspace/democratic-llm/checkpoints/llama3.1-8b-full-prism",
]

QUESTIONS_PATH = REPO_ROOT / "wvs" / "subjective_questions.json"
QUESTION_IDS = [4, 28, 61]
TOP_K = 20
MAX_NEW_TOKENS = 5
SYSTEM_PROMPT = None
HF_TOKEN = os.environ.get("HF_TOKEN")


In [3]:
def load_questions(path: Path) -> list[dict]:
    questions = json.loads(path.read_text(encoding="utf-8"))
    return sorted(questions, key=lambda row: int(row["question_id"]))


def select_questions(questions: list[dict], question_ids: list[int] | None) -> list[dict]:
    if not question_ids:
        return questions[:3]
    wanted = {int(qid) for qid in question_ids}
    selected = [q for q in questions if int(q["question_id"]) in wanted]
    missing = sorted(wanted - {int(q["question_id"]) for q in selected})
    if missing:
        raise ValueError(f"Missing question ids: {missing}")
    return selected


def load_hf_model(model_id: str, hf_token: str | None):
    tok_kwargs = {"token": hf_token}
    if "fix_mistral_regex" in inspect.signature(AutoTokenizer.from_pretrained).parameters:
        tok_kwargs["fix_mistral_regex"] = True
    tokenizer = AutoTokenizer.from_pretrained(model_id, **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model_kwargs = {"token": hf_token}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16
        model_kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()
    return model, tokenizer


def build_prompt(question: dict, system_prompt: str | None) -> str:
    option_lines = [f"({label}) {question['options'][label]}" for label in OPTION_LABELS]
    lines = []
    system_prompt = (system_prompt or "").strip()
    if system_prompt:
        lines.append(f"System: {system_prompt}")
    lines.append(f"Human: {question['question_text']}")
    lines.append("")
    lines.append("Here are the options:")
    lines.append("")
    lines.extend(option_lines)
    lines.append("")
    lines.append("Assistant: If had to select one of the options, my answer would be (")
    return "\n".join(lines)


def single_token_variants(tokenizer, label: str) -> dict[str, list[int]]:
    variants = {}
    for variant in [label, f" {label}", f"\n{label}"]:
        variants[variant] = tokenizer.encode(variant, add_special_tokens=False)
    return variants


def option_token_sets(tokenizer) -> dict[str, list[int]]:
    token_sets = {}
    for label in OPTION_LABELS:
        token_ids = set()
        for variant in [label, f" {label}", f"\n{label}"]:
            pieces = tokenizer.encode(variant, add_special_tokens=False)
            if len(pieces) == 1:
                token_ids.add(int(pieces[0]))
        if not token_ids:
            raise ValueError(
                f"Tokenizer for {getattr(tokenizer, 'name_or_path', 'model')} has no single-token encoding for {label!r}."
            )
        token_sets[label] = sorted(token_ids)
    return token_sets


def printable_token(tokenizer, token_id: int) -> str:
    token = tokenizer.decode([token_id], skip_special_tokens=False)
    return token.encode("unicode_escape").decode("utf-8")


def next_token_logits(model, tokenizer, prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = inputs.to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1].float()
    return inputs, logits


def top_next_tokens(tokenizer, logits: torch.Tensor, top_k: int, label_token_ids: dict[int, str]):
    probs = torch.softmax(logits, dim=0)
    top_probs, top_ids = torch.topk(probs, k=top_k)
    rows = []
    for rank, (prob, token_id) in enumerate(zip(top_probs.tolist(), top_ids.tolist()), start=1):
        rows.append(
            {
                "rank": rank,
                "token_id": token_id,
                "token": printable_token(tokenizer, token_id),
                "prob": prob,
                "label": label_token_ids.get(token_id),
            }
        )
    return rows


def label_probabilities(logits: torch.Tensor, option_tokens: dict[str, list[int]]) -> tuple[dict[str, float], float, dict[str, float]]:
    log_probs = torch.log_softmax(logits, dim=0)
    raw = {}
    for label in OPTION_LABELS:
        token_ids = torch.tensor(option_tokens[label], device=logits.device)
        raw[label] = float(torch.exp(torch.logsumexp(log_probs[token_ids], dim=0)).item())
    total_mass = float(sum(raw.values()))
    if total_mass > 0:
        normalized = {label: float(raw[label] / total_mass) for label in OPTION_LABELS}
    else:
        normalized = {label: 0.0 for label in OPTION_LABELS}
    return raw, total_mass, normalized


def greedy_completion(model, tokenizer, prompt: str, max_new_tokens: int) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = inputs.to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    prompt_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=False)


def inspect_model(model_id: str, questions: list[dict]):
    print(f"\n{'=' * 100}\nModel: {model_id}\n{'=' * 100}")
    model, tokenizer = load_hf_model(model_id, HF_TOKEN)
    try:
        option_tokens = option_token_sets(tokenizer)
        label_token_ids = {token_id: label for label, ids in option_tokens.items() for token_id in ids}

        print("Single-token variants used for A/B/C/D:")
        for label in OPTION_LABELS:
            variants = single_token_variants(tokenizer, label)
            print(f"  {label}: {variants}")
        print()

        for question in questions:
            prompt = build_prompt(question, SYSTEM_PROMPT)
            _, logits = next_token_logits(model, tokenizer, prompt)
            top_rows = top_next_tokens(tokenizer, logits, TOP_K, label_token_ids)
            raw_label_probs, total_mass, normalized_label_probs = label_probabilities(logits, option_tokens)
            generated = greedy_completion(model, tokenizer, prompt, MAX_NEW_TOKENS)

            print(f"Question Q{question['question_code']} ({question['question_id']}): {question['question_text']}")
            print(f"Option total probability mass: {total_mass:.6f}")
            print("Raw option probabilities:")
            for label in OPTION_LABELS:
                print(f"  {label}: {raw_label_probs[label]:.6f} -> {question['options'][label]}")
            print("Normalized option probabilities:")
            for label in OPTION_LABELS:
                print(f"  {label}: {normalized_label_probs[label]:.6f} -> {question['options'][label]}")
            print("Top next tokens:")
            for row in top_rows:
                suffix = f"  <== {row['label']} option token" if row['label'] else ""
                print(
                    f"  {row['rank']:>2}. id={row['token_id']:<6} token='{row['token']}' prob={row['prob']:.6f}{suffix}"
                )
            print(f"Greedy continuation ({MAX_NEW_TOKENS} tokens): {generated.encode('unicode_escape').decode('utf-8')}")
            print("-" * 100)
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [4]:
questions = load_questions(QUESTIONS_PATH)
selected_questions = select_questions(questions, QUESTION_IDS)
[(q["question_id"], q["question_code"], q["question_text"]) for q in selected_questions]


[(4, 'Q4', 'How important is politics in your life?'),
 (28,
  'Q28',
  'Please indicate how strongly you agree or disagree with the following statement: "When a mother works for pay, the children suffer."'),
 (61, 'Q61', 'How much do you trust people you meet for the first time?')]

In [5]:
for model_id in MODEL_IDS:
    inspect_model(model_id, selected_questions)



Model: meta-llama/Llama-3.1-8B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Single-token variants used for A/B/C/D:
  A: {'A': [32], ' A': [362], '\nA': [198, 32]}
  B: {'B': [33], ' B': [426], '\nB': [198, 33]}
  C: {'C': [34], ' C': [356], '\nC': [198, 34]}
  D: {'D': [35], ' D': [423], '\nD': [198, 35]}



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question QQ4 (4): How important is politics in your life?
Option total probability mass: 0.900646
Raw option probabilities:
  A: 0.343824 -> Very important
  B: 0.267650 -> Rather important
  C: 0.184129 -> Not very important
  D: 0.105043 -> Not at all important
Normalized option probabilities:
  A: 0.381752 -> Very important
  B: 0.297176 -> Rather important
  C: 0.204441 -> Not very important
  D: 0.116631 -> Not at all important
Top next tokens:
   1. id=32     token='A' prob=0.342509  <== A option token
   2. id=33     token='B' prob=0.266746  <== B option token
   3. id=34     token='C' prob=0.183332  <== C option token
   4. id=35     token='D' prob=0.104459  <== D option token
   5. id=883    token=' )' prob=0.007567
   6. id=2179   token='____' prob=0.007567
   7. id=36     token='E' prob=0.006678
   8. id=5235   token=' )\n\n' prob=0.006678
   9. id=64     token='a' prob=0.006678
  10. id=66     token='c' prob=0.005201
  11. id=6101   token='___' prob=0.003358
  12. id=50370 

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Single-token variants used for A/B/C/D:
  A: {'A': [32], ' A': [362], '\nA': [198, 32]}
  B: {'B': [33], ' B': [426], '\nB': [198, 33]}
  C: {'C': [34], ' C': [356], '\nC': [198, 34]}
  D: {'D': [35], ' D': [423], '\nD': [198, 35]}

Question QQ4 (4): How important is politics in your life?
Option total probability mass: 0.885910
Raw option probabilities:
  A: 0.344545 -> Very important
  B: 0.344635 -> Rather important
  C: 0.143794 -> Not very important
  D: 0.052936 -> Not at all important
Normalized option probabilities:
  A: 0.388917 -> Very important
  B: 0.389019 -> Rather important
  C: 0.162312 -> Not very important
  D: 0.059753 -> Not at all important
Top next tokens:
   1. id=33     token='B' prob=0.343143  <== B option token
   2. id=32     token='A' prob=0.343143  <== A option token
   3. id=34     token='C' prob=0.143043  <== C option token
   4. id=35     token='D' prob=0.052623  <== D option token
   5. id=36     token='E' prob=0.009144
   6. id=883    token=' )' prob=0

## What To Look For

- If `A/B/C/D` token ids show up in the top next-token list, the model can directly answer in the format the evaluator expects.
- Even if a specific raw token like `'A'` is not top-1, the aggregated option probabilities can still be high because the evaluator sums over single-token variants like `' A'` and `'\nA'`.
- If the greedy continuation starts with prose instead of a letter, that suggests the model is not well-calibrated for this exact prompt format.
- If needed, try the same cell with both a raw base model and a trained checkpoint to compare how much instruction tuning changed the next-token distribution.
